# Results for model: qwen/qwen2.5-coder-7b-instruct

In [ ]:
import polars as pl
import xgboost as xgb
from sklearn.metrics import mean_squared_error

# Load the data
data = pl.read_parquet('./jane_street_data/train.parquet')

# Feature Engineering
TOP_QUANTILE = 0.15

# Calculate the rolling batch indices
data = data.with_column(pl.date_id.cast(pl.Int32))

# Function to calculate the top quantile for each rolling batch
def calculate_top_quantile(group):
    top_quantile_value = group.sort('feature_00', reverse=True)['feature_00'].item(TOP_QUANTILE * len(group))
    return pl.Series([top_quantile_value] * len(group), name='top_quantile')

# Apply the function to each rolling batch of date_id
data = data.groupby('date_id').apply(calculate_top_quantile)

# Merge the top quantile back to the original data
data = data.unnest('top_quantile')

# Prepare data for modeling
features = [f'feature_{i:02d}' for i in range(100)]
train_data = data.filter(pl.col('month') <= 4).drop('responder_6')
test_data = data.filter(pl.col('month') > 4).drop('responder_6')

X_train = train_data[features].to_pandas()
y_train = train_data['responder_6'].to_pandas()
X_test = test_data[features].to_pandas()

# Train the XGBoost Regressor
model = xgb.XGBRegressorobjective='reg:squarederror', random_state=42)
model.fit(X_train, y_train)

# Predict on the test data
y_pred = model.predict(X_test)

# Calculate the RMSE
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f'RMSE (Test): {rmse}')